# 00c - LISFLOOD domain masks on the ERA5 0.25 deg grid and on the native EC-Earth3 grid

**Purpose**: build, once, the mask that is applied to every 0.25 deg output (GCM "ERA5-like" after the downscaling,
ERA5 for the validation, later the RCM), so that every dataset is compared on exactly the same pixels and LISFLOOD
gets all the pixels it needs.

**Input**: the GloFAS / LISFLOOD domain `template_Global_03min.nc` (0.05 deg, 1 = inside the domain, NaN = outside;
90N-60S, no Antarctica; built on the MERIT Hydro river network), downloaded from the JRC open-data FTP:
`CEMS-GLOFAS/LISFLOOD_static_and_parameter_maps_for_GloFAS/v2.1.1_OS-LISFLOOD-v5.x/Main/`, stored in `extra_files/`.

**Rule ("at least one")**: an ERA5 pixel is inside the domain if **at least one** of the 0.05 deg LISFLOOD pixels whose
centre falls inside it is inside the domain (about 5 x 5 LISFLOOD pixels per ERA5 pixel). Every LISFLOOD pixel of the
domain is therefore covered, plus a thin coastal margin.

**Then the same rule from ERA5 to EC-Earth3**: an EC-Earth3 pixel (0.70 deg, Gaussian `gr` grid, shared by the
historical run and all the hosing runs) is inside if **at least one** of the ERA5 pixels whose centre falls inside it is
inside. Every ERA5 pixel of the domain therefore lies in an EC-Earth3 pixel of the domain.

**Outputs** in `extra_files/` (grids as read by the pipeline: ascending latitude, longitude -180..180):

| file | grid | to apply to |
|---|---|---|
| `lisflood_domain_on_ERA5_025.nc` | ERA5 0.25 deg | ERA5, GCM "ERA5-like" 0.25 deg after the downscaling, validation (later the RCM) |
| `lisflood_domain_on_EC-Earth3_gr.nc` | EC-Earth3 native | EC-Earth3 historical and hosing runs at native resolution (exploration) |
| `lisflood_domain_buffer_on_ERA5_025.nc` | ERA5 0.25 deg | ERA5 mask grown by 1 pixel (0.25 deg) in every direction |
| `lisflood_domain_buffer_on_EC-Earth3_gr.nc` | EC-Earth3 native | the buffered ERA5 mask transferred to EC-Earth3 ("at least one" rule) |
| `lisflood_domain_buffer_EC-Earth3_footprint_on_ERA5_025.nc` | ERA5 0.25 deg | ERA5 pixels whose centre falls in a pixel of the buffered EC-Earth3 mask |

The three `buffer` files are for a downscaling **restricted to land + a coastal margin** (if computing over the
oceans is too heavy): apply `ecearth3_buffer` to EC-Earth3 and `ecearth3_buffer_on_era5` to ERA5, so that the GCM
pixels and the ERA5 pixels cover **exactly the same area**, with every LISFLOOD pixel at least 0.25 deg inside it
(margin for interpolating the forcing onto the 0.05 deg LISFLOOD grid).

Each file has `domain` (1 = inside, 0 = outside) and `domain_frac` (share of the finer pixels that are inside, information only).
Load them with `ba_utils.load_domain_mask(cfg, grid=...)`, grid = `"era5"`, `"ecearth3"`, `"era5_buffer"`,
`"ecearth3_buffer"` or `"ecearth3_buffer_on_era5"`.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings, datetime
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
cfg = bu.load_config(CONFIG_PATH)

EXTRA = bu.domain_mask_path(cfg, "era5").parent
LISFLOOD_FILE = EXTRA / "template_Global_03min.nc"
OUT_FILE = bu.domain_mask_path(cfg, "era5")
print("input :", LISFLOOD_FILE, "| exists:", LISFLOOD_FILE.exists())
print("output:", OUT_FILE)

## 1. Inputs: LISFLOOD domain (0.05 deg) and ERA5 grid (0.25 deg)

In [ ]:
lis = xr.open_dataset(LISFLOOD_FILE)
lis = bu.norm_coords(lis[list(lis.data_vars)[0]]).load()       # ascending lat, lon -180..180
fine = (lis == 1).values
print(f"LISFLOOD: {lis.sizes['lat']} x {lis.sizes['lon']}, lat {float(lis.lat.min()):.3f}..{float(lis.lat.max()):.3f}, "
      f"lon {float(lis.lon.min()):.3f}..{float(lis.lon.max()):.3f} | pixels in the domain: {fine.sum():,}")

# ERA5 grid: only the coordinates of ONE ERA5 file are needed (all years share the same grid),
# normalised as the pipeline reads it (same coordinates as every ERA5 / ERA5-like field)
ERA5_FILE = bu._pattern_files(cfg["datasets"]["era5"]["patterns"]["pr"])[0]
era5 = bu.norm_coords(xr.open_dataset(ERA5_FILE)["pr"].isel(time=0, drop=True))
print("ERA5 grid taken from:", ERA5_FILE)

# EC-Earth3 native grid: one file of the historical run, checked against one file of every hosing run
def first_grid(name):
    f = bu._pattern_files(cfg["datasets"][name]["patterns"]["tasmax"])[0]
    return bu.norm_coords(xr.open_dataset(f)["tasmax"].isel(time=0, drop=True)), f

gcm, GCM_FILE = first_grid("gcm_historical")
print("EC-Earth3 grid taken from:", GCM_FILE)
for name in ("gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"):
    if bu.dataset_available(cfg, name, "tasmax"):
        assert bu.same_grid(first_grid(name)[0], gcm), f"{name} is not on the EC-Earth3 historical grid"
print(f"EC-Earth3: {gcm.sizes['lat']} x {gcm.sizes['lon']} (same grid for the historical and the hosing runs)")
print(f"ERA5    : {era5.sizes['lat']} x {era5.sizes['lon']}, lat {float(era5.lat.min())}..{float(era5.lat.max())}, "
      f"lon {float(era5.lon.min())}..{float(era5.lon.max())}")

## 2. Transfer to the ERA5 grid (rule: at least one LISFLOOD pixel inside)

In [ ]:
mask, frac = bu.mask_any(fine, lis.lat.values, lis.lon.values, era5.lat.values, era5.lon.values)
domain = xr.DataArray(mask.astype("int8"), dims=("lat", "lon"), coords={"lat": era5.lat, "lon": era5.lon}, name="domain")
domain_frac = xr.DataArray(frac.astype("float32"), dims=("lat", "lon"), coords={"lat": era5.lat, "lon": era5.lon}, name="domain_frac")

# checks: every LISFLOOD pixel of the domain falls in an ERA5 pixel of the mask
def globe_share(m, lat, res):
    """Share of the area of the whole sphere covered by the True cells of a regular grid of step `res` deg."""
    return float((np.cos(np.deg2rad(lat))[:, None] * m).sum() * np.deg2rad(res) ** 2 / (4 * np.pi))

area_era5 = globe_share(mask, era5.lat.values, 0.25)
area_lis = globe_share(fine, lis.lat.values, 0.05)
iy = np.searchsorted(bu._edges(era5.lat.values), lis.lat.values) - 1
xe = bu._edges(era5.lon.values)
ix = np.clip(np.searchsorted(xe, ((lis.lon.values - xe[0]) % 360) + xe[0]) - 1, 0, era5.sizes["lon"] - 1)
r, c = np.nonzero(fine)
missed = int((~mask[iy[r], ix[c]]).sum())
print(f"ERA5 pixels in the domain: {mask.sum():,} ({100 * area_era5:.1f} % of the globe area)")
print(f"LISFLOOD domain           : {100 * area_lis:.1f} % of the globe area (the ERA5 mask is a bit larger: coastal margin)")
print(f"LISFLOOD pixels of the domain NOT covered by the ERA5 mask: {missed} (must be 0)")
assert missed == 0

## 3. Save

In [ ]:
out = xr.Dataset({"domain": domain, "domain_frac": domain_frac})
out["domain"].attrs = {"long_name": "LISFLOOD (GloFAS) domain on the ERA5 0.25 deg grid", "flag_values": "0 1",
                       "flag_meanings": "outside inside"}
out["domain_frac"].attrs = {"long_name": "share of the 0.05 deg LISFLOOD pixels of the ERA5 pixel that are in the domain",
                            "units": "1"}
out.attrs = {"source": str(LISFLOOD_FILE.name) + " (GloFAS LISFLOOD static maps v2.1.1_OS-LISFLOOD-v5.x, Main/)",
             "rule": "ERA5 pixel inside if at least one LISFLOOD pixel whose centre falls in it is inside",
             "grid": "ERA5 0.25 deg as read by ba_utils.read_raw (ascending lat, lon -180..179.75)",
             "created": datetime.date.today().isoformat(), "created_by": "notebook 00c_land_mask"}
tmp = OUT_FILE.with_suffix(".tmp.nc")
out.to_netcdf(tmp, encoding={"domain": {"zlib": True}, "domain_frac": {"zlib": True}})
tmp.replace(OUT_FILE)
print("written:", OUT_FILE)
m = bu.load_domain_mask(cfg, grid="era5")
print("reload check:", int(m.sum()), "pixels | same grid as ERA5:", bu.same_grid(m, era5))

## 3b. Transfer from ERA5 to the native EC-Earth3 grid (rule: at least one ERA5 pixel inside)

In [ ]:
mask_gcm, frac_gcm = bu.mask_any(mask, era5.lat.values, era5.lon.values, gcm.lat.values, gcm.lon.values)
domain_gcm = xr.DataArray(mask_gcm.astype("int8"), dims=("lat", "lon"), coords={"lat": gcm.lat, "lon": gcm.lon}, name="domain")

# check: every ERA5 pixel of the domain falls in an EC-Earth3 pixel of the domain
ye, xe = bu._edges(gcm.lat.values), bu._edges(gcm.lon.values)
iy = np.searchsorted(ye, era5.lat.values) - 1
ix = np.clip(np.searchsorted(xe, ((era5.lon.values - xe[0]) % 360) + xe[0]) - 1, 0, gcm.sizes["lon"] - 1)
r, c = np.nonzero(mask)
missed_gcm = int((~mask_gcm[iy[r], ix[c]]).sum())
wg = np.cos(np.deg2rad(gcm.lat.values))[:, None]
area_gcm = float((wg * mask_gcm).sum() / (wg * np.ones_like(mask_gcm)).sum())
print(f"EC-Earth3 pixels in the domain: {mask_gcm.sum():,} of {mask_gcm.size:,} ({100 * area_gcm:.1f} % of the globe area)")
print(f"ERA5 pixels of the domain NOT covered by the EC-Earth3 mask: {missed_gcm} (must be 0)")
assert missed_gcm == 0

out_g = xr.Dataset({"domain": domain_gcm,
                    "domain_frac": xr.DataArray(frac_gcm.astype("float32"), dims=("lat", "lon"),
                                                coords={"lat": gcm.lat, "lon": gcm.lon})})
out_g["domain"].attrs = {"long_name": "LISFLOOD (GloFAS) domain on the native EC-Earth3 grid", "flag_values": "0 1",
                         "flag_meanings": "outside inside"}
out_g["domain_frac"].attrs = {"long_name": "share of the ERA5 0.25 deg pixels of the EC-Earth3 pixel that are in the domain",
                              "units": "1"}
out_g.attrs = {**out.attrs, "rule": "EC-Earth3 pixel inside if at least one ERA5 pixel whose centre falls in it is inside "
                                    "(ERA5 mask from the LISFLOOD 0.05 deg domain with the same rule)",
               "grid": "EC-Earth3 'gr' (Gaussian N128) as read by ba_utils.read_raw (ascending lat, lon -180..180)"}
OUT_GCM = bu.domain_mask_path(cfg, "ecearth3")
tmp = OUT_GCM.with_suffix(".tmp.nc")
out_g.to_netcdf(tmp, encoding={"domain": {"zlib": True}, "domain_frac": {"zlib": True}})
tmp.replace(OUT_GCM)
print("written:", OUT_GCM)
mg = bu.load_domain_mask(cfg, grid="ecearth3")
print("reload check:", int(mg.sum()), "pixels | same grid as EC-Earth3:", bu.same_grid(mg, gcm))

## 3c. Masks with a 0.25 deg buffer (for a downscaling restricted to land + coastal margin)

1. ERA5 mask + 1 ERA5 pixel in every direction (8 neighbours, periodic in longitude);
2. transferred to EC-Earth3 with the "at least one" rule;
3. footprint of (2) on ERA5: the ERA5 pixels whose centre falls in a pixel of (2), i.e. (2) "applied to ERA5".

In [ ]:
def save_mask(m, lat, lon, grid_key, long_name, rule, grid, frac=None):
    ds = xr.Dataset({"domain": xr.DataArray(np.asarray(m).astype("int8"), dims=("lat", "lon"), coords={"lat": lat, "lon": lon})})
    ds["domain"].attrs = {"long_name": long_name, "flag_values": "0 1", "flag_meanings": "outside inside"}
    if frac is not None:
        ds["domain_frac"] = xr.DataArray(frac.astype("float32"), dims=("lat", "lon"), coords={"lat": lat, "lon": lon})
    ds.attrs = {**out.attrs, "rule": rule, "grid": grid}
    path = bu.domain_mask_path(cfg, grid_key)
    tmp = path.with_suffix(".tmp.nc")
    ds.to_netcdf(tmp, encoding={v: {"zlib": True} for v in ds.data_vars})
    tmp.replace(path)
    print("written:", path.name)

# 1. ERA5 + 0.25 deg buffer
mask_b = bu.dilate(mask, 1)
save_mask(mask_b, era5.lat, era5.lon, "era5_buffer", "LISFLOOD domain + 0.25 deg buffer on the ERA5 0.25 deg grid",
          "ERA5 LISFLOOD-domain mask grown by 1 ERA5 pixel (8 neighbours, periodic in longitude)",
          "ERA5 0.25 deg as read by ba_utils.read_raw")
# 2. -> EC-Earth3 ("at least one")
mask_gb, frac_gb = bu.mask_any(mask_b, era5.lat.values, era5.lon.values, gcm.lat.values, gcm.lon.values)
save_mask(mask_gb, gcm.lat, gcm.lon, "ecearth3_buffer", "LISFLOOD domain + 0.25 deg buffer on the native EC-Earth3 grid",
          "EC-Earth3 pixel inside if at least one pixel of the buffered ERA5 mask has its centre in it",
          "EC-Earth3 'gr' (Gaussian N128) as read by ba_utils.read_raw", frac_gb)
# 3. footprint of (2) on ERA5 = (2) applied to ERA5
mask_fb = bu.mask_footprint(mask_gb, gcm.lat.values, gcm.lon.values, era5.lat.values, era5.lon.values)
save_mask(mask_fb, era5.lat, era5.lon, "ecearth3_buffer_on_era5",
          "Footprint on the ERA5 grid of the buffered EC-Earth3 LISFLOOD-domain mask",
          "ERA5 pixel inside if its centre falls in a pixel of the buffered EC-Earth3 mask",
          "ERA5 0.25 deg as read by ba_utils.read_raw")

# checks
share = lambda m, lat: float((np.cos(np.deg2rad(lat))[:, None] * m).sum() / (np.cos(np.deg2rad(lat))[:, None] * np.ones_like(m)).sum())
print(f"ERA5 mask            : {mask.sum():8,} px ({100 * share(mask, era5.lat.values):.1f} % of the globe)")
print(f"ERA5 + buffer        : {mask_b.sum():8,} px ({100 * share(mask_b, era5.lat.values):.1f} %)")
print(f"EC-Earth3 + buffer   : {mask_gb.sum():8,} px ({100 * share(mask_gb, gcm.lat.values):.1f} %)")
print(f"its footprint on ERA5: {mask_fb.sum():8,} px ({100 * share(mask_fb, era5.lat.values):.1f} %)")
assert (mask_b >= mask).all(), "the buffered mask must contain the mask"
assert (mask_fb >= mask_b).all(), "the footprint must contain the buffered ERA5 mask"
mg_b = bu.mask_any(mask_fb, era5.lat.values, era5.lon.values, gcm.lat.values, gcm.lon.values)[0]
assert (mg_b == mask_gb).all(), "footprint and EC-Earth3 mask must describe the same area"
print("checks ok: mask <= buffer <= footprint, and footprint <-> EC-Earth3 buffer mask cover the same area")

## 4. Control maps

Grey = outside, colour = inside, red lines = real coastlines (GISCO 1:1 million, no administrative borders). Check in particular the large lakes (Caspian Sea, Great Lakes, Lake Victoria),
the coasts, small islands and Greenland.

In [ ]:
def show(ax, da, title, extent=None):
    d = da if extent is None else da.sel(lat=slice(extent[2], extent[3]), lon=slice(extent[0], extent[1]))
    ax.imshow(np.where(np.isfinite(d.values.astype(float)), d.values.astype(float), np.nan), origin="lower",
              extent=[float(d.lon.min()), float(d.lon.max()), float(d.lat.min()), float(d.lat.max())],
              cmap="viridis", vmin=0, vmax=1, interpolation="nearest", aspect="auto")
    bu.draw_borders(ax, BORDERS, color="r", lw=0.4)      # real coastlines (GISCO) over the mask
    ax.set_xlim(float(d.lon.min()), float(d.lon.max())); ax.set_ylim(float(d.lat.min()), float(d.lat.max()))
    ax.set_facecolor("0.85"); ax.set_title(title, fontsize=9)

BORDERS = bu.borders(cfg, "01M")          # coastlines only (Eurostat GISCO, finest resolution)

lis_plot = lis.where(lis == 1)                                  # NaN outside
dom_plot = domain.where(domain == 1).astype(float)
gcm_plot = domain_gcm.where(domain_gcm == 1).astype(float)
fig, axes = plt.subplots(1, 3, figsize=(24, 4.5))
show(axes[0], lis_plot[::5, ::5], "LISFLOOD domain (0.05 deg, shown 1 pixel out of 5)")
show(axes[1], dom_plot, "LISFLOOD domain on the ERA5 0.25 deg grid")
show(axes[2], gcm_plot, "LISFLOOD domain on the EC-Earth3 grid")
plt.tight_layout(); plt.savefig(EXTRA / "lisflood_domain_masks_global.png", dpi=110); plt.show()

ZOOMS = {"Europe": (-25, 45, 30, 72), "Caspian Sea": (45, 56, 36, 48), "Great Lakes": (-93, -75, 40, 50),
         "Lake Victoria": (30, 36, -4, 2), "Greenland": (-75, -10, 58, 84)}
fig, axes = plt.subplots(3, len(ZOOMS), figsize=(4.2 * len(ZOOMS), 10.5))
for c, (lab, ext) in enumerate(ZOOMS.items()):
    show(axes[0, c], lis_plot, f"{lab}: LISFLOOD 0.05 deg", ext)
    show(axes[1, c], dom_plot, f"{lab}: on ERA5 0.25 deg", ext)
    show(axes[2, c], gcm_plot, f"{lab}: on EC-Earth3", ext)
plt.tight_layout(); plt.savefig(EXTRA / "lisflood_domain_masks_zooms.png", dpi=110); plt.show()

# buffered masks (Europe and the western Mediterranean)
f = lambda m, ref: xr.DataArray(np.where(m, 1.0, np.nan), dims=("lat", "lon"), coords={"lat": ref.lat, "lon": ref.lon})
panels = [("ERA5 mask", f(mask, era5)), ("ERA5 + 0.25 deg buffer", f(mask_b, era5)),
          ("EC-Earth3 + buffer", f(mask_gb, gcm)), ("its footprint on ERA5 (to apply to ERA5)", f(mask_fb, era5))]
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for r, ext in enumerate([(-25, 45, 30, 72), (-10, 20, 34, 46)]):
    for c, (lab, da) in enumerate(panels):
        show(axes[r, c], da, lab, ext)
plt.tight_layout(); plt.savefig(EXTRA / "lisflood_domain_buffer_masks.png", dpi=110); plt.show()

## 5. How to use it

```python
m = bu.load_domain_mask(cfg, grid="era5")        # ERA5 0.25 deg grid (True = inside)
x_land = x.where(m)                              # x: ERA5, GCM ERA5-like 0.25 deg, ...
mg = bu.load_domain_mask(cfg, grid="ecearth3")   # native EC-Earth3 grid (historical and hosing runs)
g_land = g.where(mg)                             # g: native EC-Earth3 field

# downscaling restricted to land + coastal margin (same area on both grids):
gcm_in  = g.where(bu.load_domain_mask(cfg, grid="ecearth3_buffer"))
era5_in = x.where(bu.load_domain_mask(cfg, grid="ecearth3_buffer_on_era5"))
```
The downscaling itself runs on the whole globe for now; the masks are applied to its outputs and to the validation.